In [26]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
warnings.filterwarnings('ignore')

In [55]:
class SpamRNN_LSTM:
    def __init__(self, file_path, max_words=5000, max_len=100):
        self.file_path = file_path
        self.max_words = max_words
        self.max_len = max_len
        self.tokenizer = Tokenizer(num_words=self.max_words, oov_token="<OOV>")
        self.model = None
        self.df = None

    def load_data(self):

        print("--- Loading Dataset ---")

        self.df = pd.read_csv(self.file_path, sep=',', encoding='latin-1')


        self.df = self.df[['v1', 'v2']]
        self.df.columns = ['label', 'text']

        print(f"Dataset loaded. Unique labels: {self.df['label'].unique()}")

    def preprocess_data(self):

        self.load_data()

        self.df['label'] = self.df['label'].astype(str).str.strip().str.lower()
        self.df['label'] = self.df['label'].map({'ham': 0, 'spam': 1})

        self.df = self.df.dropna(subset=['label'])


        self.df = self.df.sample(frac=1).reset_index(drop=True)

        self.tokenizer.fit_on_texts(self.df['text'].astype(str).values)
        sequences = self.tokenizer.texts_to_sequences(self.df['text'].astype(str).values)


        self.X = pad_sequences(sequences, maxlen=self.max_len, padding='pre')
        self.y = self.df['label'].values.astype('float32')


    def build_model(self):

        self.preprocess_data()

        vocab_size = len(self.tokenizer.word_index) + 1

        inputs = tf.keras.Input(shape=(self.max_len,))

        x = layers.Embedding(vocab_size, 32)(inputs)


        x = layers.LSTM(32)(x)

        outputs = layers.Dense(1, activation='sigmoid')(x)

        self.model = Model(inputs, outputs)

        opt = tf.keras.optimizers.Adam(learning_rate=0.001)
        self.model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])


    def train_model(self):

        self.build_model()

        if self.model:


            counts = np.bincount(self.y.astype(int))
            total = len(self.y)

            class_weight = {
                0: (1 / counts[0]) * (total / 2.0),
                1: (1 / counts[1]) * (total / 2.0)
            }

            print(f"Applying Class Weights: {class_weight}")


            self.model.fit(
                self.X,
                self.y,
                epochs=5,
                batch_size=32,
                validation_split=0.2,
                class_weight=class_weight
            )

    def run_predictions(self, messages):

        if self.model is None:
              self.train_model()

        if self.model:
            for msg in messages:
                seq = self.tokenizer.texts_to_sequences([msg])
                padded = pad_sequences(seq, maxlen=self.max_len, padding='post')
                prob = self.model.predict(padded, verbose=0)
                print(f"[{'SPAM' if prob > 0.5 else 'HAM'}] -> {msg}")


In [58]:
if __name__ == "__main__":
    spam_app = SpamRNN_LSTM("/content/spam.csv")
    spam_app.run_predictions(["Congratulations on your new job!",
        "Hey, can we meet at 5pm?" ])

--- Loading Dataset ---
Dataset loaded. Unique labels: ['ham' 'spam']
Applying Class Weights: {0: np.float64(0.5774093264248704), 1: np.float64(3.72958500669344)}
Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.8930 - loss: 0.3677 - val_accuracy: 0.9668 - val_loss: 0.1367
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.9809 - loss: 0.0891 - val_accuracy: 0.9874 - val_loss: 0.0685
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.9892 - loss: 0.0439 - val_accuracy: 0.9857 - val_loss: 0.0582
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - accuracy: 0.9957 - loss: 0.0234 - val_accuracy: 0.9848 - val_loss: 0.0579
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.9978 - loss: 0.0121 - val_accuracy: 0.9848 - val_loss: 0.0622
[HAM] -> Congratulations on your new job!
[HAM] -> Hey, can we meet at 5pm?
